# Great Expectations
Great Expectations (GX) is an open‑source Python framework for data quality validation. It’s used to define rules (“expectations”) about your data, automatically check them in pipelines, and produce clear reports so teams can trust their analytics, machine learning, and production systems.

# Example mini project

# Step 1: Generate a "Perfect" Synthetic Dataset




In [ ]:
import pandas as pd
import numpy as np

# Set a seed for reproducibility
np.random.seed(42)
n_samples = 1000

# Generate synthetic data with clear rules
data = {
    'employee_id': range(1, n_samples + 1),
    'experience_years': np.random.randint(0, 40, n_samples),
    'salary': np.random.randint(30000, 150000, n_samples),
    'department': np.random.choice(['Sales', 'Engineering', 'HR', 'Marketing'], n_samples)
}
df = pd.DataFrame(data)

# Introduce a rule: Salary must be at least $1000 per year of experience
df['salary'] = df.apply(lambda row: max(row['salary'], row['experience_years'] * 1000), axis=1)

df.to_csv('perfect_employee_data.csv', index=False)

In [ ]:
df.head()

,employee_id,experience_years,salary,department
0,1,38,74327,Sales
1,2,28,98904,Marketing
2,3,14,33797,HR
3,4,7,77882,Sales
4,5,20,43718,Engineering


# Step 2: Create a "Faulty" Dataset to Test Against

We, intentionally break the rules in a few rows of a new dataset. This will show GX's validation capabilities.

In [ ]:
# Copy the perfect dataset and intentionally introduce errors
df_faulty = df.copy()

# Error 1: Set an invalid negative value for experience
df_faulty.loc[5, 'experience_years'] = -5

# Error 2: Set a salary that is too low for the employee's experience
df_faulty.loc[10, 'salary'] = 500  # Salary is only $500, but experience is likely >0

df_faulty.to_csv('faulty_employee_data.csv', index=False)

# Step 3: Validate with Great Expectations

Now, we'll write a simple GX script to define our expectations based on the perfect_employee_data.csv and then validate the faulty_employee_data.csv against those expectations .


In [ ]:
!pip install great_expectations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.2 MB/s eta 0:00:00


In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np

# 1. Create the synthetic datasets
np.random.seed(42)
n_samples = 1000

df_perfect = pd.DataFrame({
    'employee_id': range(1, n_samples + 1),
    'experience_years': np.random.randint(0, 40, n_samples),
    'salary': np.random.randint(30000, 150000, n_samples),
    'department': np.random.choice(['Sales', 'Engineering', 'HR', 'Marketing'], n_samples)
})

df_perfect['salary'] = df_perfect.apply(
    lambda row: max(row['salary'], row['experience_years'] * 1000), axis=1
)

#df_faulty = df_perfect.copy()
#df_faulty.loc[5, 'experience_years'] = -5
#df_faulty.loc[10, 'salary'] = 500



# 2. Setup Great Expectations context
context = gx.get_context(mode="ephemeral") # creates a temporary, in-memory Great Expectations environment.
# In Gx, the context is the central object that manages the entire GX environment
# 3. Create a Data Source and asset
data_source = context.data_sources.add_pandas("pandas_datasource") # register a Pandas datasource inside the context
data_asset = data_source.add_dataframe_asset(name="employee_asset") # defines a logical dataset (asset) within the pandas data source for validation

# 4. Build batch definitions (GX v1 uses batch definitions, not add_batch)
batch_definition = data_asset.add_batch_definition_whole_dataframe("my_batch_definition")# “Defines a batch configuration that uses the entire DataFrame as the dataset for validation.”

# 5. Create expectation suite
suite = gx.ExpectationSuite(name="employee_validation_suite") #In Gx, a suite is simply a container for expectations.
suite = context.suites.add(suite) # This registers the suite with our GX context

# 6. Define expectations directly on the suite (no validator needed for setup)
#pulling in specific expectation classes from Great Expectations:
from great_expectations.expectations import (
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToBeInSet,
)


#Here we  attach expectations (rules) to our suite
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="employee_id"))
suite.add_expectation(ExpectColumnValuesToBeBetween(column="experience_years", min_value=0, max_value=40))
suite.add_expectation(ExpectColumnValuesToBeBetween(column="salary", min_value=30000, max_value=150000))
suite.add_expectation(ExpectColumnValuesToBeInSet(column="department", value_set=['Sales', 'Engineering', 'HR', 'Marketing']))

# 7. Create a Validation Definition
# This constructs a validation definition object
#It ties together:
  #- A batch definition (the actual slice of data you want to validate).
  #- An expectation suite (the set of rules you want to apply).

validation_def = context.validation_definitions.add(

        gx.ValidationDefinition(
        name="employee_validation",
        data=batch_definition,
        suite=suite,
    )
)

# 8. Validate perfect data
perfect_batch = batch_definition.get_batch(batch_parameters={"dataframe": df_perfect})
perfect_results = validation_def.run(batch_parameters={"dataframe": df_perfect})
print(f"Perfect data — Validation successful: {perfect_results['success']}")



INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpb2qeny0j' for ephemeral docs site


Calculating Metrics:   0%|          | 0/32 [00:00<?, ?it/s]

Perfect data — Validation successful: True


In [ ]:
# 9. Validate faulty data
faulty_results = validation_def.run(batch_parameters={"dataframe": df_faulty})
print(f"\nFaulty data — Validation successful: {faulty_results['success']}")

if not faulty_results['success']:
    print("\nFailed expectations:")
    for result in faulty_results['results']:
        if not result['success']:
            exp_type = result['expectation_config']['type']
            col = result['expectation_config']['kwargs']['column']
            print(f"  - {exp_type} on column '{col}'")

Calculating Metrics:   0%|          | 0/32 [00:00<?, ?it/s]


Faulty data — Validation successful: False

Failed expectations:
  - expect_column_values_to_be_between on column 'experience_years'
  - expect_column_values_to_be_between on column 'salary'
